In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from joblib import dump
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, matthews_corrcoef, confusion_matrix

In [2]:
param_grid = {
    "n_estimators": [100, 300, 500], 
    "criterion": ["gini", "entropy"],           
    "min_samples_split": [2, 5, 10],           
    "min_samples_leaf": [1, 2, 4],              
    "max_features": ["auto", "sqrt", "log2"],  
    "max_depth": [None, 10, 20, 30],             
    "bootstrap": [True, False]  
}

In [3]:
def undersampling(df_data, seed):
    ten_percent=df_data[df_data["target"]==2].sample(frac=0.10, random_state=42)
    X=df_data.drop('target', axis=1)
    y=df_data['target']  
    #Se definen los objetos para submuestrear
    X['index']=X.index 
    undersampler=RandomUnderSampler(sampling_strategy='not minority', random_state=42)    

    #Se aplica el submuestreo
    X_res, y_res=undersampler.fit_resample(X, y)
    df_resampled=pd.concat([X_res,y_res], axis=1)

    index_res=X_res['index']

    mask=~X['index'].isin(index_res)
    excluded_data=df_data[mask.values]
    data_independent= pd.concat([ten_percent, excluded_data], axis=0)
    data_independent.reset_index(drop=True, inplace=True)
    data_independent.to_csv("../../models/data/data_independent.csv", index=False)
    
    return df_resampled

In [4]:
def oversampling(df_data, seed):
    X = df_data.drop('target', axis=1)
    y = df_data['target']  
    #Se definen los objetos para sobremuestrear
    smote = SMOTE(random_state=seed)

    #Se aplica el sobremuestreo
    X_res, y_res= smote.fit_resample(X, y)
    df_resampled = pd.concat([X_res,y_res], axis=1)
    
    return df_resampled

In [5]:
def split(df_data, seed):
    #Separa los datos
    data_under= undersampling(df_data, seed)
    data_over= oversampling(df_data, seed)
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    train_data_under, val_data_under = train_test_split(data_under, test_size=0.2, random_state=seed)
    train_data_over, val_data_over = train_test_split(data_over, test_size=0.2, random_state=seed)
    return train_data, val_data, train_data_under, val_data_under, train_data_over, val_data_over

In [6]:
def metrics(predict_val, y_val, dataset, div):
    acc_value = accuracy_score(y_pred=predict_val, y_true=y_val) 
    recall_value = recall_score(y_pred=predict_val, y_true=y_val, average='weighted')
    precision_value = precision_score(y_pred=predict_val, y_true=y_val, average='weighted') 
    f1_value = f1_score(y_pred=predict_val, y_true=y_val, average='weighted')
    mcc_value = matthews_corrcoef(y_pred=predict_val, y_true=y_val)
    cm = confusion_matrix(y_pred=predict_val, y_true=y_val)
    cm_df = pd.DataFrame(cm)
    cm_dict = cm_df.to_dict()

    df_metrics = pd.DataFrame([[dataset, "Random_Forest", div, acc_value, recall_value, precision_value, f1_value, mcc_value, cm_dict]],
                              columns=["dataset", "model", "sampling", "acc", "recall", "precision", "f1", "mcc", "conf_matrix"])

    return df_metrics

In [7]:
def train(train, val, div, seed):
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()

    print(f"Train Random Forest with seed {seed} and division {div}")
    results = []
    rf = RandomForestClassifier(random_state=seed)
    rf.fit(X_train, y_train)

    dump(rf, f"../../models/RandomForest_Grid_{seed}_{div}.joblib")

    y_pred_train = rf.predict(X_train)
    y_pred_val = rf.predict(X_val)

    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/RandomForest_Grid_{seed}_{div}_predictions.csv", index=False)
    
    train_metrics = metrics(y_pred_train, y_train, "Train", div)
    val_metrics = metrics(y_pred_val, y_val, "Validation", div)
    results.append(train_metrics)
    results.append(val_metrics)
    return pd.concat(results, ignore_index=True), rf

In [8]:
def grid_function(rf, train, val, seed, div): 
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    df_combined = val.copy()

    print(f"GridSearchCV for Random Forest with seed {seed} and division {div}")
    grid = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    best_params = grid.best_params_

    print(f"Best estimators: {best_model}")
    print(f"Best parameters found: {best_params}")
    best_params = json.dumps(best_params, indent=4)
    with open(f"../../models/RandomForest_Grid_{seed}_{div}_best_params.json", "w") as f:
        f.write(best_params)
    dump(best_model, f"../../models/RandomForest_Grid_{seed}_{div}_best.joblib")

    y_pred_train = best_model.predict(X_train)
    y_pred_val = best_model.predict(X_val)

    df_combined.drop(columns="target", inplace=True)
    df_combined["y_true"] = y_val
    df_combined["y_pred"] = y_pred_val
    df_combined.to_csv(f"../../models/data/RandomForest_Grid_{seed}_{div}_best_predictions.csv", index=False)

    train_metrics = metrics(y_pred_train, y_train, "Train", div)
    val_metrics = metrics(y_pred_val, y_val, "Validation", div)
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    return results

In [9]:
def main_train(df_data, seed, grid_search=False):
    all_metrics = []
    all_metrics_grid = []
    df_train, df_val, df_train_under, df_val_under, df_train_over, df_val_over = split(df_data, seed)
    metrics_orig, model_orig = train(df_train, df_val, "Original", seed)
    metrics_under, model_under = train(df_train_under, df_val_under, "Under", seed)
    metrics_over, model_over = train(df_train_over, df_val_over, "Over", seed)

    all_metrics = pd.concat([metrics_orig, metrics_under, metrics_over], ignore_index=True)
    all_metrics.to_csv(f"../../metrics/RandomForest_Grid_{seed}_metrics.csv", index=False)
    if grid_search:
        all_metrics_grid = pd.concat([grid_function(model_orig, df_train, df_val, seed, "Original"),
                                      grid_function(model_under, df_train_under, df_val_under, seed, "Under"),
                                      grid_function(model_over, df_train_over, df_val_over, seed, "Over")], ignore_index=True)
        all_metrics_grid.to_csv(f"../../metrics/RandomForest_Grid_{seed}_grid_metrics.csv", index=False)
    

In [10]:
repr_name="ProtT5"
df_data = pd.read_csv(f"../../data/numerical_rep/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

In [11]:
folder = "../../data/numerical_rep/"
seed= 42

In [12]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_{repr_name}.csv"
main_train(df_data, seed, grid_search=True)
print(f"Finished processing {repr_name}")
print("=====================================")

Processing ProtT5
Train Random Forest with seed 42 and division Original
Train Random Forest with seed 42 and division Under
Train Random Forest with seed 42 and division Over
GridSearchCV for Random Forest with seed 42 and division Original


/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
2160 fits failed out of a total of 6480.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1755 fits failed with the following error:
Traceback (most recent call last):
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py", line 1466, in wrapper
    estimator._validate_params()
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py", line 666, in _validate_params
    valid

Best estimators: RandomForestClassifier(bootstrap=False, criterion='entropy', max_depth=10,
                       min_samples_leaf=4, random_state=42)
Best parameters found: {'bootstrap': False, 'criterion': 'entropy', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 100}
GridSearchCV for Random Forest with seed 42 and division Under


/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
2160 fits failed out of a total of 6480.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
794 fits failed with the following error:
Traceback (most recent call last):
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py", line 1466, in wrapper
    estimator._validate_params()
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/base.py", line 666, in _validate_params
    valida

Best estimators: RandomForestClassifier(criterion='entropy', min_samples_split=10,
                       n_estimators=500, random_state=42)
Best parameters found: {'bootstrap': True, 'criterion': 'entropy', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 500}
GridSearchCV for Random Forest with seed 42 and division Over


/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
2160 fits failed out of a total of 6480.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1116 fits failed with the following error:
Traceback (most recent call last):
  File "/home/jmartin/micromamba/envs/ML/lib/python3.9/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)

Best estimators: RandomForestClassifier(criterion='entropy', max_depth=20, n_estimators=500,
                       random_state=42)
Best parameters found: {'bootstrap': True, 'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
Finished processing ProtT5
